<a href="https://colab.research.google.com/github/dwililiya07/cifar10-architecture-experiments/blob/main/Proper_code_of_transfer_learning_in_cifar_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Description

This project aims to implement transfer learning method, use the pretrained model resnet18 for image classification.

The dataset used is CIFAR-10, which contains 60.000 RGB images (32x32 pixels) across 10 different classes.

I finetuned baseline model to fit with CIFAR10 dataset. The best model at the end got through experimentation. After the get the results, we'll compare with the resulf of the model that build from scratch.

This project will be developed using Python and PyTorch.



In [1]:
# import library and device
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Experiment 1 - Baseline Model with Data Augmentation

In [2]:
# Data Transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [3]:
# dataset and dataloader
train_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform_test
)

train_loader = DataLoader(train_dataset, batch_size=64,
                          shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64,
                          shuffle=False, num_workers=2, pin_memory=True)

100%|██████████| 170M/170M [00:03<00:00, 43.1MB/s]


note: commonly number of worker = number of cpu cores/2

In [ ]:
# model setup
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# freeze backbone
for param in model.parameters():
  param.requires_grad = False

# replace classifier
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

# set criterion and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

In [ ]:
# print how many parameters are trainable
sum(p.numel() for p in model.parameters() if p.requires_grad)

5130

In [ ]:
# verify that only the classifier is trainable
for name, param in model.named_parameters():
  if param.requires_grad:
    print(name)

fc.weight
fc.bias


In [4]:
# Training function
def train_one_epoch(model, loader, optimizer, criterion, device):

    model.train()

    for m in model.modules():
      if isinstance(m, nn.BatchNorm2d):
        m.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()               # zero gradient
        outputs = model(images)             # forward pass
        loss = criterion(outputs, labels)   # calculate the loss
        loss.backward()                     # backward pass
        optimizer.step()                    # update weights

        # accumulation loss and accuracy
        total_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)


    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples

    return avg_loss, avg_acc

In [5]:
# evaluation function
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples

    return avg_loss, avg_acc

In [ ]:
num_epochs = 20
patience = 3
best_acc = 0
epochs_without_improvement = 0

# main training loop
for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    test_loss, test_acc = evaluate(
        model, test_loader, criterion, device
    )

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    # Early Stopping
    if test_acc > best_acc:
      best_acc = test_acc
      epochs_without_improvement=0

      torch.save({
          'model_state_dict': model.state_dict(),
          'test_acc': test_acc,
      }, "best_model_tf_cifar_1.pth")
    else:
      epochs_without_improvement += 1
      print(f"No Improvement for {epochs_without_improvement} epoch(s)")

    if epochs_without_improvement >= patience:
      print("Early stopping triggered")
      break
    print("-" * 50)

Epoch 1/20
Train Loss: 0.5993 | Train Acc: 0.7938
Test Loss: 0.5855 | Test Acc: 0.8016
--------------------------------------------------
Epoch 2/20
Train Loss: 0.5963 | Train Acc: 0.7921
Test Loss: 0.5830 | Test Acc: 0.8029
--------------------------------------------------
Epoch 3/20
Train Loss: 0.5883 | Train Acc: 0.7971
Test Loss: 0.5837 | Test Acc: 0.8020
No Improvement for 1 epoch(s)
--------------------------------------------------
Epoch 4/20
Train Loss: 0.5930 | Train Acc: 0.7952
Test Loss: 0.6023 | Test Acc: 0.7953
No Improvement for 2 epoch(s)
--------------------------------------------------
Epoch 5/20
Train Loss: 0.5869 | Train Acc: 0.7965
Test Loss: 0.5933 | Test Acc: 0.7977
No Improvement for 3 epoch(s)
Early stopping triggered


Gap train-test sangat kecil, sehingga hampir tidak ada overfitting. Namun angka lossnya antar epoch sangat sedikit perbedaannya, bahkan training berhenti di epoch 5 karena tidak ada peningkatan accurasi 3 epoch berturut-turut. it feels there was something wrong in training process. maybe the architecure of model doesn't get the pattern to learn it well. Is that because we only unfreeze classifier layer?

## Experiment 2 - Unfreeze layer4

In [6]:
# model setup
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# freeze backbone
for param in model.parameters():
  param.requires_grad = False

# unfreeze layer4
for param in model.layer4.parameters():
  param.requires_grad = True

# replace classifier
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

# set criterion and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 187MB/s]


In [7]:
num_epochs = 20
patience = 3
best_acc = 0
epochs_without_improvement = 0

# main training loop
for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    test_loss, test_acc = evaluate(
        model, test_loader, criterion, device
    )

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    # Early Stopping
    if test_acc > best_acc:
      best_acc = test_acc
      epochs_without_improvement=0

      torch.save({
          'model_state_dict': model.state_dict(),
          'test_acc': test_acc,
      }, "best_model_tf_cifar_2.pth")
    else:
      epochs_without_improvement += 1
      print(f"No Improvement for {epochs_without_improvement} epoch(s)")

    if epochs_without_improvement >= patience:
      print("Early stopping triggered")
      break
    print("-" * 50)

Epoch 1/20
Train Loss: 0.4041 | Train Acc: 0.8588
Test Loss: 0.3638 | Test Acc: 0.8754
--------------------------------------------------
Epoch 2/20
Train Loss: 0.2400 | Train Acc: 0.9180
Test Loss: 0.2316 | Test Acc: 0.9206
--------------------------------------------------
Epoch 3/20
Train Loss: 0.1832 | Train Acc: 0.9359
Test Loss: 0.2264 | Test Acc: 0.9227
--------------------------------------------------
Epoch 4/20
Train Loss: 0.1513 | Train Acc: 0.9483
Test Loss: 0.2802 | Test Acc: 0.9160
No Improvement for 1 epoch(s)
--------------------------------------------------
Epoch 5/20
Train Loss: 0.1245 | Train Acc: 0.9563
Test Loss: 0.2299 | Test Acc: 0.9293
--------------------------------------------------
Epoch 6/20
Train Loss: 0.1023 | Train Acc: 0.9643
Test Loss: 0.2856 | Test Acc: 0.9183
No Improvement for 1 epoch(s)
--------------------------------------------------
Epoch 7/20
Train Loss: 0.0876 | Train Acc: 0.9693
Test Loss: 0.2278 | Test Acc: 0.9319
-------------------------

Unfreeze the layer before classifier layer is such good things. The model can learn well, because layer4 is high level semantics. then the model adapt with 10 output classes. i think that's enough to increase the accuration significantly in cifar10 dataset.

## Experiment 3 - use the different learning rate

In [8]:
# model setup
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# freeze backbone
for param in model.parameters():
  param.requires_grad = False

# unfreeze layer4
for param in model.layer4.parameters():
  param.requires_grad = True

# replace classifier
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

# set criterion and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3}
])

In [9]:
num_epochs = 20
patience = 3
best_acc = 0
epochs_without_improvement = 0

# main training loop
for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    test_loss, test_acc = evaluate(
        model, test_loader, criterion, device
    )

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    # Early Stopping
    if test_acc > best_acc:
      best_acc = test_acc
      epochs_without_improvement=0

      torch.save({
          'model_state_dict': model.state_dict(),
          'test_acc': test_acc,
      }, "best_model_tf_cifar_3.pth")
    else:
      epochs_without_improvement += 1
      print(f"No Improvement for {epochs_without_improvement} epoch(s)")

    if epochs_without_improvement >= patience:
      print("Early stopping triggered")
      break
    print("-" * 50)

Epoch 1/20
Train Loss: 0.3968 | Train Acc: 0.8611
Test Loss: 0.2631 | Test Acc: 0.9104
--------------------------------------------------
Epoch 2/20
Train Loss: 0.2363 | Train Acc: 0.9190
Test Loss: 0.2454 | Test Acc: 0.9173
--------------------------------------------------
Epoch 3/20
Train Loss: 0.1840 | Train Acc: 0.9361
Test Loss: 0.2360 | Test Acc: 0.9193
--------------------------------------------------
Epoch 4/20
Train Loss: 0.1488 | Train Acc: 0.9476
Test Loss: 0.2670 | Test Acc: 0.9183
No Improvement for 1 epoch(s)
--------------------------------------------------
Epoch 5/20
Train Loss: 0.1268 | Train Acc: 0.9551
Test Loss: 0.2160 | Test Acc: 0.9341
--------------------------------------------------
Epoch 6/20
Train Loss: 0.1023 | Train Acc: 0.9642
Test Loss: 0.2336 | Test Acc: 0.9299
No Improvement for 1 epoch(s)
--------------------------------------------------
Epoch 7/20
Train Loss: 0.0875 | Train Acc: 0.9695
Test Loss: 0.2837 | Test Acc: 0.9189
No Improvement for 2 epoc

the third experiment is the best configuration because it achieves the same performance with fewer epochs and better optimization structure.

## Conclusion of Comparing the Model

Using transfer learning is way better than build the model from scratch. Especially, when solve the case that has big dataset. in transfer learning, we can adjust pretrained model's layer and finetuning them to match with our datasets.

But for beginner like me, understanding when build the model from the scratch hits different. Learning layer by layer, the function like optimizer, regularization, criterion, etc help me understand why and how.

Actually, I learn deep learning like 1 year ago in bootcamp. But at that time, i can't understand why. like why we use resize in that dataset but not in this dataset or how to make this model not overfit. In 2026, I learn it again more deeply, no rush, trying be patience. Yap, that's all.